In [1]:
import tensorflow as tf
import os

# Configuración del Architect
BUCKET_NAME = "sentinel-vision-data-v1" 
GCS_PATH = f"gs://{BUCKET_NAME}/data/*/*.jpg"

# 1. Crear una lista de todos los archivos
list_ds = tf.data.Dataset.list_files(GCS_PATH, shuffle=True)

# 2. Extraer etiquetas de las rutas
# Las carpetas en GCS tienen el formato: gs://bucket/data/CLASS_NAME/image.jpg
def get_label(file_path):
    parts = tf.strings.split(file_path, os.path.sep)
    # El penúltimo elemento es el nombre de la carpeta (la clase)
    return parts[-2]

# 3. Función de pre-procesamiento core
def process_path(file_path):
    label = get_label(file_path)
    # Leer el archivo de GCS
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    # Redimensionar para la red (Sentinel-2 es 64x64, pero escalaremos a 224x224 para Transfer Learning)
    img = tf.image.resize(img, [224, 224])
    # Normalización: $x_{norm} = \frac{x}{255.0}$
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

# Aplicar el mapeo en paralelo
train_ds = list_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)

2026-04-27 01:34:32.306283: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-27 01:34:36.451349: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64:/usr/lib/x86_64-linux-gnu/:/opt/conda/lib
2026-04-27 01:34:36.451508: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local

In [2]:
from tensorflow.keras.layers import StringLookup

# 1. Obtener la lista de nombres de carpetas directamente de GCS
# Listamos solo el primer nivel dentro de /data/
import subprocess

# Comando de sistema para listar carpetas en el bucket
cmd = f"gcloud storage ls gs://{BUCKET_NAME}/data/"
output = subprocess.check_output(cmd, shell=True).decode("utf-8")

# Limpiamos los nombres (quitamos rutas y slashes)
class_names = [line.split('/')[-2] for line in output.splitlines() if line.endswith('/')]
class_names.sort()

print(f"Clases detectadas: {class_names}")

# 2. Crear la capa de mapeo
# mask_token=None porque no estamos trabajando con secuencias de texto
label_encoder = StringLookup(vocabulary=class_names, mask_token=None)

# 3. Función actualizada para procesar datos
def process_path_with_label(file_path):
    # Extraer texto de la ruta
    label_text = get_label(file_path)
    # Convertir texto a número usando la capa
    label_idx = label_encoder(label_text)
    
    # Procesar imagen (reutilizamos la lógica anterior)
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    img = tf.cast(img, tf.float32) / 255.0
    
    return img, label_idx

# 4. Re-crear el dataset final
train_ds = list_ds.map(process_path_with_label, num_parallel_calls=tf.data.AUTOTUNE)

Clases detectadas: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


In [3]:
BATCH_SIZE = 32

# Aplicamos el empaquetado y el prefetch
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

print("Pipeline configurado y optimizado.")

Pipeline configurado y optimizado.


In [4]:
for images, labels in train_ds.take(1):
    print(f"Forma del lote de imágenes: {images.shape}")
    print(f"Etiquetas del lote: {labels.numpy()}")

Forma del lote de imágenes: (32, 224, 224, 3)
Etiquetas del lote: [ 1  3  1  3  4  8  2  7  2  5  5  8 10  6  1  4 10  1  8  8  8  1  3  4
  2  2  7  7  8  5  9  3]
